In [25]:
from unittest import case

import requests
from bs4 import BeautifulSoup
import pandas as pd
import plotly.express as px


#print('Hello')

In [33]:
URL="https://finance.yahoo.com/markets/crypto/most-active/"
#print(URL)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

def parse_human_numbers(num_str):
    if num_str is None:
        return None
    num_str = num_str.replace(',', '')
    last_char = num_str[-1]
    num_str_no_last_char = num_str[:-1]
    match (last_char):
        case 'K':
            return float(num_str_no_last_char) * 1_000
        case 'M':
            return float(num_str_no_last_char) * 1_000_000
        case 'B':
            return float(num_str_no_last_char) * 1_000_000_000
        case 'T':
            return float(num_str_no_last_char) * 1_000_000_000_000
        case _:
            return float(num_str)


def get_crypto_volumes():
    response = requests.get(URL, headers=HEADERS)
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table', attrs={'class':'yf-19xszjc'})
    #print(table)
    rows = table.find('tbody').find_all('tr')

    data = []
    for row in rows:
        #print(row)
        cols = row.find_all('td')
        symbol = cols[0].text.strip()
        name = cols[1].text.strip()
        price = parse_human_numbers(cols[4].text.strip())
        volume = parse_human_numbers(cols[8].text.strip())
        market_cap = parse_human_numbers(cols[6].text.strip())
        #print(symbol, name, price, volume, market_cap)
        #print('-'*60)

        data.append({
            'symbol': symbol,
            'name': name,
            'price': price,
            'volume': volume,
            'market_cap': market_cap
        })
    return pd.DataFrame(data)

def save_crypto_volumes(df):
    df.to_csv('data/crypto_volumes.csv', index=False)
    df.to_json('data/crypto_volumes.json', orient='records', indent=4)


def plot_volume(df):
    df_sorted = df.sort_values('volume', ascending=False)
    df_sorted_big = df_sorted[df_sorted['volume'] > 1_000_000_000]
    df_sorted_big['market capit'] = df_sorted_big['market_cap'].apply(lambda x: str(x/1_000_000_000) + 'B')

    fig = px.bar(df_sorted_big,
                 x='symbol',
                 y='volume',
                 title='Crypto Volumes - Yahoo Finanse, Most Active',
                 hover_data=['name', 'price', 'market capit']
                )
    fig.show()


df = get_crypto_volumes()
save_crypto_volumes(df)
plot_volume(df)


## Example showing web scraping with bs4 and pandas
'''print('hello')'''